In [19]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report
import tensorflow as tf
import joblib
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import f1_score, precision_score, recall_score
# Load data
df = pd.read_csv("diabetes.csv")

In [2]:
# 2) Feature engineering
# Replace invalid zeros with NaN 
zero_cols = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]
df[zero_cols] = df[zero_cols].replace(0, np.nan)

In [3]:
df["BMI_Age"] = df["BMI"] / (df["Age"] + 1)
df["Glucose_Insulin"] = df["Glucose"] * df["Insulin"]
df["BP_Age"] = df["BloodPressure"] / (df["Age"] + 1)

In [4]:
X = df.drop("Outcome", axis=1)
y = df["Outcome"]

In [5]:
# 4) Outlier handling 
def cap_outliers_iqr(df, cols):
    df = df.copy()
    for c in cols:
        q1 = df[c].quantile(0.25)
        q3 = df[c].quantile(0.75)
        iqr = q3 - q1
        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr
        df[c] = df[c].clip(lower, upper)
    return df

X = cap_outliers_iqr(X, X.columns)

In [12]:
# pipline and transformer
numeric_features = X.columns

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features)
    ]
)

In [13]:
# 6) Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Fit preprocessing on train only
X_train_prep = preprocessor.fit_transform(X_train)
X_test_prep = preprocessor.transform(X_test)

In [15]:
# 7) Handle class imbalance
neg, pos = np.bincount(y_train)
total = neg + pos
class_weight = {
    0: total / (2 * neg),
    1: total / (2 * pos)
}

In [16]:
# 8) Model
model = tf.keras.Sequential([
    tf.keras.layers.Dense(32, activation="relu", input_shape=(X_train_prep.shape[1],)),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(16, activation="relu"),
    tf.keras.layers.Dense(1, activation="sigmoid")
])

model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

history = model.fit(
    X_train_prep, y_train,
    validation_split=0.2,
    epochs=50,
    batch_size=32,
    class_weight=class_weight,
    verbose=1
)

C:\Users\SAEEDCOMPUTERS\anaconda3\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/50
16/16 ━━━━━━━━━━━━━━━━━━━━ 4s 45ms/step - accuracy: 0.5886 - loss: 0.7185 - val_accuracy: 0.6341 - val_loss: 0.6874
Epoch 2/50
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.6253 - loss: 0.6758 - val_accuracy: 0.6585 - val_loss: 0.6565
Epoch 3/50
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.6782 - loss: 0.6489 - val_accuracy: 0.6585 - val_loss: 0.6307
Epoch 4/50
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.6782 - loss: 0.6234 - val_accuracy: 0.7073 - val_loss: 0.6044
Epoch 5/50
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step - accuracy: 0.7088 - loss: 0.5945 - val_accuracy: 0.6911 - val_loss: 0.5782
Epoch 6/50
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.7291 - loss: 0.5874 - val_accuracy: 0.6911 - val_loss: 0.5524
Epoch 7/50
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 0.7149 - loss: 0.5689 - val_accuracy: 0.6992 - val_loss: 0.5321
Epoch 8/50
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.7413 - loss: 0.5435 - val_accuracy: 0.7154 - v

In [21]:
# 9) Evaluate
y_pred = (model.predict(X_test_prep) > 0.5).astype(int)
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))


5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step
Accuracy: 0.7662337662337663
              precision    recall  f1-score   support

           0       0.87      0.75      0.81       100
           1       0.63      0.80      0.70        54

    accuracy                           0.77       154
   macro avg       0.75      0.77      0.76       154
weighted avg       0.79      0.77      0.77       154



In [22]:
# probabilities
y_prob = model.predict(X_test_prep).ravel()

# apply threshold 0.55
threshold = 0.55
y_pred = (y_prob >= threshold).astype(int)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step
Accuracy: 0.7792207792207793
              precision    recall  f1-score   support

           0       0.87      0.78      0.82       100
           1       0.66      0.78      0.71        54

    accuracy                           0.78       154
   macro avg       0.76      0.78      0.77       154
weighted avg       0.79      0.78      0.78       154



In [23]:
# Save model
model.save("diabetes_model.keras")

# Save preprocessor
joblib.dump(preprocessor, "diabetes_preprocessor.pkl")

# Save threshold
threshold = 0.55
with open("diabetes_threshold.json", "w") as f:
    json.dump({"threshold": threshold}, f)

print("Saved: diabetes_model.keras, diabetes_preprocessor.pkl, diabetes_threshold.json")

Saved: diabetes_model.keras, diabetes_preprocessor.pkl, diabetes_threshold.json
